# PipelineWatch-NG — Module 6: Publication-Quality Figures
**Run Modules 1–3 and 2b first, then open this notebook.**

---

## Overview

This module generates two types of publication-ready outputs:

1. **Static high-resolution maps** — cartopy + Natural Earth data
   - Coastlines, country borders, rivers, lakes
   - Labeled latitude/longitude gridlines
   - Scale bar, north arrow, inset locator map
   - Works for ANY region in the world — just change the bounding box

2. **Time series charts** — monthly FIRMS fire counts + TROPOMI SO2
   - 12-month baseline vs recent comparison
   - Cluster-level SO2 time series for confirmed sites
   - Suitable for paper Figure 3 / Figure 4

## Why cartopy?

cartopy uses Natural Earth vector data (free, CC0 license) for coastlines,
borders, rivers and lakes. The projection system handles any region on Earth
automatically — the same code works over Niger Delta, North Sea, Gulf of Mexico,
or Southeast Asia with no changes except the bounding box coordinates.

---

## Install dependencies (run once)

```bash
conda install -c conda-forge cartopy -y
pip install matplotlib numpy pandas
```

---

## Outputs

| File | Description |
|------|-------------|
| outputs/fig1_study_area.png | Full TNP corridor overview map |
| outputs/fig2_cluster_detail.png | Zoomed cluster locations with SO2 confirmation |
| outputs/fig3_timeseries.html | Interactive monthly fire + SO2 time series |
| outputs/fig4_cluster_timeseries.html | Per-cluster SO2 evolution over time |


## Cell 1 — Setup

In [1]:
import matplotlib
matplotlib.use('Agg')  # non-interactive backend
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
from matplotlib_scalebar.scalebar import ScaleBar
from IPython.display import Image, IFrame, display

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER

import ee
import os
import json
import gc
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime

GEE_PROJECT_ID = 'pipelinewatch-ng'
try:
    ee.Initialize(project=GEE_PROJECT_ID)
    print('GEE connected: ' + str(ee.Number(42).getInfo()))
except Exception:
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT_ID)

CACHE_DIR  = '../data/cached'
OUTPUT_DIR = '../outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Setup complete.')


C:\Users\taylo\anaconda3\envs\pipelinewatch\lib\site-packages\pyproj\network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)
C:\Users\taylo\anaconda3\envs\pipelinewatch\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


GEE connected: 42
Setup complete.


## Cell 2 — Load all cached outputs

In [2]:
# Load cluster data
df_clusters = pd.read_csv(os.path.join(CACHE_DIR, 'm2_refinery_clusters.csv'))
df_val      = pd.read_csv(os.path.join(CACHE_DIR, 'm2b_tropomi_validation.csv'))
df_risk     = pd.read_csv(os.path.join(CACHE_DIR, 'm3_risk_scored.csv'))

# Load fire hotspots
with open(os.path.join(CACHE_DIR, 'fire_hotspots.geojson')) as f:
    firms_gj = json.load(f)
df_firms = pd.DataFrame([
    {'lon': feat['geometry']['coordinates'][0],
     'lat': feat['geometry']['coordinates'][1],
     **feat['properties']}
    for feat in firms_gj['features']
])

# Merge validation results into cluster table
df_clusters = df_clusters.merge(
    df_val[['cluster_id','SO2_mean_DU','SO2_max_DU','SO2_confirmed','verdict']],
    on='cluster_id', how='left'
)

print('Clusters: ' + str(len(df_clusters)))
print('Confirmed: ' + str(df_clusters['SO2_confirmed'].sum()))
print('Fire hotspots: ' + str(len(df_firms)))
print('Risk pixels: ' + str(len(df_risk)))


Clusters: 8
Confirmed: 6
Fire hotspots: 50
Risk pixels: 270


## Cell 3 — Figure 1: Study area overview map

**Globally portable:** Change `ROI_BOUNDS`, `REGION_NAME`, and `COUNTRY` to map
any pipeline corridor in the world. The cartopy Natural Earth features
(coastlines, borders, rivers, lakes) are available globally.

**Resolution options:** `'10m'` = highest detail (slow), `'50m'` = balanced, `'110m'` = fast overview

In [3]:
# Load Natural Earth features at 50m — lighter on memory than 10m
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import cartopy.io.img_tiles as cimgt
import gc

RESOLUTION = '50m'   # 50m is publication-acceptable and much lighter than 10m

OCEAN      = cfeature.NaturalEarthFeature('physical','ocean',     RESOLUTION, facecolor='#D6EAF8', edgecolor='none')
LAND       = cfeature.NaturalEarthFeature('physical','land',      RESOLUTION, facecolor='#F5F5F0', edgecolor='none')
LAKES      = cfeature.NaturalEarthFeature('physical','lakes',     RESOLUTION, facecolor='#AED6F1', edgecolor='none')
RIVERS     = cfeature.NaturalEarthFeature('physical','rivers_lake_centerlines', RESOLUTION, facecolor='none', edgecolor='#AED6F1', linewidth=0.8)
BORDERS    = cfeature.NaturalEarthFeature('cultural','admin_0_countries',       RESOLUTION, facecolor='none', edgecolor='#555555', linewidth=0.6)
STATES     = cfeature.NaturalEarthFeature('cultural','admin_1_states_provinces',RESOLUTION, facecolor='none', edgecolor='#AAAAAA', linewidth=0.4)
COASTLINE  = cfeature.NaturalEarthFeature('physical','coastline', RESOLUTION, facecolor='none', edgecolor='#1A5276', linewidth=1.0)

gc.collect()
print('Features loaded at ' + RESOLUTION + ' resolution.')

Features loaded at 50m resolution.


In [4]:
import plotly.graph_objects as go

# ── Config ────────────────────────────────────────────────────────────────────
ROI_BOUNDS  = [6.50, 5.00, 7.20, 5.80]   # [west, south, east, north]
REGION_NAME = 'Trans Niger Pipeline Corridor'

# ── Base map ──────────────────────────────────────────────────────────────────
fig = go.Figure()

# Study area bounding box
box_lons = [ROI_BOUNDS[0], ROI_BOUNDS[2], ROI_BOUNDS[2], ROI_BOUNDS[0], ROI_BOUNDS[0]]
box_lats = [ROI_BOUNDS[1], ROI_BOUNDS[1], ROI_BOUNDS[3], ROI_BOUNDS[3], ROI_BOUNDS[1]]
fig.add_trace(go.Scattergeo(
    lon=box_lons, lat=box_lats, mode='lines',
    line=dict(color='#185FA5', width=2, dash='dash'),
    name='TNP corridor', showlegend=True
))

# VIIRS fire hotspots
if not df_firms.empty:
    fig.add_trace(go.Scattergeo(
        lon=df_firms['lon'], lat=df_firms['lat'],
        mode='markers',
        marker=dict(color='#FAC775', size=6, opacity=0.7,
                    line=dict(color='white', width=0.3)),
        name='VIIRS fire hotspot',
        text=['T21=' + str(round(t,1)) + 'K  count=' + str(round(c,1))
              for t, c in zip(df_firms['T21_max_K'], df_firms['fire_count'])],
        hovertemplate='%{text}<extra>VIIRS hotspot</extra>'
    ))

# Unconfirmed clusters
unconf = df_clusters[df_clusters['SO2_confirmed'] == False]
if not unconf.empty:
    fig.add_trace(go.Scattergeo(
        lon=unconf['centroid_lon'], lat=unconf['centroid_lat'],
        mode='markers+text',
        marker=dict(color='#378ADD', size=14, symbol='circle',
                    line=dict(color='white', width=1)),
        text=['C' + str(int(r['cluster_id'])) for _, r in unconf.iterrows()],
        textposition='top right', textfont=dict(size=9, color='#1A1A1A'),
        name='Unconfirmed cluster (2/8)',
        customdata=[[str(int(r['cluster_id'])), str(round(r['max_T21_K'],1)),
                     str(int(r['n_hotspots'])), str(round(r.get('SO2_max_DU',0),3))]
                    for _, r in unconf.iterrows()],
        hovertemplate='<b>Cluster %{customdata[0]}</b><br>'
                      'T21 max: %{customdata[1]} K<br>'
                      'Hotspots: %{customdata[2]}<br>'
                      'SO2 max: %{customdata[3]} DU<extra></extra>'
    ))

# Confirmed clusters
conf = df_clusters[df_clusters['SO2_confirmed'] == True]
if not conf.empty:
    fig.add_trace(go.Scattergeo(
        lon=conf['centroid_lon'], lat=conf['centroid_lat'],
        mode='markers+text',
        marker=dict(color='#E24B4A', size=18, symbol='star',
                    line=dict(color='white', width=1)),
        text=['C' + str(int(r['cluster_id'])) for _, r in conf.iterrows()],
        textposition='top right', textfont=dict(size=9, color='#E24B4A',
                                                 family='Arial Black'),
        name='SO2-confirmed refinery (6/8)',
        customdata=[[str(int(r['cluster_id'])), str(round(r['max_T21_K'],1)),
                     str(int(r['n_hotspots'])), str(round(r.get('SO2_max_DU',0),3)),
                     str(r.get('verdict',''))]
                    for _, r in conf.iterrows()],
        hovertemplate='<b>Cluster %{customdata[0]} — CONFIRMED</b><br>'
                      'T21 max: %{customdata[1]} K<br>'
                      'Hotspots: %{customdata[2]}<br>'
                      'SO2 max: %{customdata[3]} DU<br>'
                      '%{customdata[4]}<extra></extra>'
    ))

# HIGH risk zones from ML
high_risk = df_risk[df_risk['risk_tier'] == 'HIGH']
if not high_risk.empty:
    fig.add_trace(go.Scattergeo(
        lon=high_risk['lon'], lat=high_risk['lat'],
        mode='markers',
        marker=dict(color='#E24B4A', size=10, symbol='diamond',
                    line=dict(color='#7B241C', width=1), opacity=0.9),
        name='HIGH risk zone (ML)',
        text=['Score=' + str(round(s,3)) + '  VV=' + str(round(v,2)) + 'dB'
              for s, v in zip(high_risk['combined_risk_score'], high_risk['VV'])],
        hovertemplate='%{text}<extra>HIGH risk</extra>'
    ))

# ── Layout — natural geography basemap with coastlines, rivers, borders ───────
fig.update_geos(
    scope='africa',
    center=dict(lat=5.40, lon=6.85),
    projection_type='mercator',
    lataxis_range=[ROI_BOUNDS[1]-0.3, ROI_BOUNDS[3]+0.3],
    lonaxis_range=[ROI_BOUNDS[0]-0.3, ROI_BOUNDS[2]+0.3],
    showcoastlines=True,  coastlinecolor='#1A5276', coastlinewidth=1.5,
    showland=True,        landcolor='#F5F5F0',
    showocean=True,       oceancolor='#D6EAF8',
    showlakes=True,       lakecolor='#AED6F1',
    showrivers=True,      rivercolor='#AED6F1', riverwidth=1.0,
    showcountries=True,   countrycolor='#555555', countrywidth=0.8,
    showsubunits=True,    subunitcolor='#AAAAAA', subunitwidth=0.5,
    lonaxis=dict(showgrid=True, gridwidth=0.5, gridcolor='rgba(150,150,150,0.4)',
                 tick0=6.0, dtick=0.2),
    lataxis=dict(showgrid=True, gridwidth=0.5, gridcolor='rgba(150,150,150,0.4)',
                 tick0=5.0, dtick=0.2),
    bgcolor='white',
    framecolor='#333333', framewidth=1,
    resolution=50,
)

fig.update_layout(
    title=dict(
        text='PipelineWatch-NG: ' + REGION_NAME + '<br>'
             '<sup>Multi-sensor crude oil theft detection | 6/8 clusters SO\u2082-confirmed | '
             'Niger Delta, Nigeria</sup>',
        font=dict(size=14), x=0.05
    ),
    height=650,
    paper_bgcolor='white',
    legend=dict(x=0.01, y=0.01, bgcolor='rgba(255,255,255,0.9)',
                bordercolor='#CCCCCC', borderwidth=1, font=dict(size=10))
)

out1 = os.path.join(OUTPUT_DIR, 'fig1_study_area.html')
fig.write_html(out1)
print('Saved: ' + out1)
display(IFrame(src=out1, width='100%', height=680))

Saved: ../outputs\fig1_study_area.html


## Cell 4 — Figure 2: Cluster detail maps

One zoomed panel per confirmed cluster — shows the exact location with
coastlines, rivers, and labeled coordinates. Suitable for paper supplementary
figures or a conference poster cluster panel.

In [5]:
# Plotly cluster detail panels — renders in browser, zero kernel memory
from plotly.subplots import make_subplots

confirmed_clusters = df_clusters[df_clusters['SO2_confirmed'] == True].reset_index(drop=True)
n_conf = len(confirmed_clusters)
ZOOM_RADIUS = 0.25

ncols = 3
nrows = (n_conf + ncols - 1) // ncols

fig = make_subplots(
    rows=nrows, cols=ncols,
    specs=[[{'type': 'geo'} for _ in range(ncols)] for _ in range(nrows)],
    subplot_titles=['Cluster ' + str(int(r['cluster_id'])) +
                    ' | SO2 max: ' + str(round(r.get('SO2_max_DU',0),2)) + ' DU' +
                    ' | T21: ' + str(round(r['max_T21_K'],0)) + 'K'
                    for _, r in confirmed_clusters.iterrows()],
    horizontal_spacing=0.02,
    vertical_spacing=0.08
)

for i, (_, row) in enumerate(confirmed_clusters.iterrows()):
    r = i // ncols + 1
    c = i % ncols + 1
    clat = row['centroid_lat']
    clon = row['centroid_lon']
    geo_key = 'geo' if (r == 1 and c == 1) else 'geo' + str(i + 1)

    # VIIRS hotspots in zoom window
    nearby = df_firms[
        (df_firms['lat'].between(clat - ZOOM_RADIUS, clat + ZOOM_RADIUS)) &
        (df_firms['lon'].between(clon - ZOOM_RADIUS, clon + ZOOM_RADIUS))
    ]
    if not nearby.empty:
        fig.add_trace(go.Scattergeo(
            lon=nearby['lon'], lat=nearby['lat'],
            mode='markers',
            marker=dict(color='#FAC775', size=8, opacity=0.8,
                        line=dict(color='white', width=0.5)),
            name='VIIRS hotspot', showlegend=(i == 0),
            hovertemplate='T21=%{text}K<extra>VIIRS</extra>',
            text=[str(round(t, 1)) for t in nearby['T21_max_K']]
        ), row=r, col=c)

    # HIGH risk pixels in window
    nearby_risk = df_risk[
        (df_risk['risk_tier'] == 'HIGH') &
        (df_risk['lat'].between(clat - ZOOM_RADIUS, clat + ZOOM_RADIUS)) &
        (df_risk['lon'].between(clon - ZOOM_RADIUS, clon + ZOOM_RADIUS))
    ]
    if not nearby_risk.empty:
        fig.add_trace(go.Scattergeo(
            lon=nearby_risk['lon'], lat=nearby_risk['lat'],
            mode='markers',
            marker=dict(color='#E24B4A', size=10, symbol='diamond',
                        line=dict(color='#7B241C', width=0.8), opacity=0.9),
            name='HIGH risk zone', showlegend=(i == 0),
            hovertemplate='Score=%{text}<extra>HIGH risk</extra>',
            text=[str(round(s, 3)) for s in nearby_risk['combined_risk_score']]
        ), row=r, col=c)

    # Cluster centroid star
    fig.add_trace(go.Scattergeo(
        lon=[clon], lat=[clat],
        mode='markers+text',
        marker=dict(color='#E24B4A', size=18, symbol='star',
                    line=dict(color='white', width=1)),
        text=['C' + str(int(row['cluster_id']))],
        textposition='top right',
        textfont=dict(size=10, color='#E24B4A', family='Arial Black'),
        name='Cluster centroid', showlegend=(i == 0),
        hovertemplate='<b>C' + str(int(row['cluster_id'])) + '</b><br>' +
                      str(round(clat, 4)) + 'N, ' + str(round(clon, 4)) + 'E<extra></extra>'
    ), row=r, col=c)

    # Update geo for this subplot
    fig.update_layout(**{
        geo_key: dict(
            scope='africa',
            center=dict(lat=clat, lon=clon),
            projection_type='mercator',
            lataxis_range=[clat - ZOOM_RADIUS, clat + ZOOM_RADIUS],
            lonaxis_range=[clon - ZOOM_RADIUS, clon + ZOOM_RADIUS],
            showcoastlines=True,  coastlinecolor='#1A5276', coastlinewidth=1.5,
            showland=True,        landcolor='#F5F5F0',
            showocean=True,       oceancolor='#D6EAF8',
            showlakes=True,       lakecolor='#AED6F1',
            showrivers=True,      rivercolor='#AED6F1', riverwidth=1.2,
            showcountries=True,   countrycolor='#555555', countrywidth=0.8,
            showsubunits=True,    subunitcolor='#AAAAAA', subunitwidth=0.5,
            lonaxis=dict(showgrid=True, gridwidth=0.5,
                         gridcolor='rgba(150,150,150,0.4)', dtick=0.1),
            lataxis=dict(showgrid=True, gridwidth=0.5,
                         gridcolor='rgba(150,150,150,0.4)', dtick=0.1),
            bgcolor='white', resolution=50,
        )
    })

# Hide unused panels
for j in range(n_conf, nrows * ncols):
    r = j // ncols + 1
    c = j % ncols + 1
    geo_key = 'geo' + str(j + 1)
    fig.update_layout(**{geo_key: dict(visible=False)})

fig.update_layout(
    title=dict(
        text='PipelineWatch-NG — SO2-Confirmed Cluster Detail Maps<br>'
             '<sup>Trans Niger Pipeline corridor | Oct-Dec 2023 | '
             'Star = cluster centroid | Diamond = HIGH risk zone</sup>',
        font=dict(size=13), x=0.05
    ),
    height=380 * nrows,
    paper_bgcolor='white',
    showlegend=True,
    legend=dict(x=0.01, y=-0.05, orientation='h',
                bgcolor='rgba(255,255,255,0.9)',
                bordercolor='#CCCCCC', borderwidth=1)
)

out2 = os.path.join(OUTPUT_DIR, 'fig2_cluster_detail.html')
fig.write_html(out2)
print('Saved: ' + out2)
display(IFrame(src=out2, width='100%', height=400 * nrows))

Saved: ../outputs\fig2_cluster_detail.html


## Cell 5 — Fetch monthly time series data from GEE

GEE call only — no plotting in this cell.
Fetches monthly FIRMS fire counts and TROPOMI SO2 mean for
the full Jan 2023 – Dec 2024 period (24 months).

In [6]:
ROI = ee.Geometry.Rectangle([6.50, 5.00, 7.20, 5.80])

MONTHS = [
    ('2023-01','2023-01-01','2023-02-01'), ('2023-02','2023-02-01','2023-03-01'),
    ('2023-03','2023-03-01','2023-04-01'), ('2023-04','2023-04-01','2023-05-01'),
    ('2023-05','2023-05-01','2023-06-01'), ('2023-06','2023-06-01','2023-07-01'),
    ('2023-07','2023-07-01','2023-08-01'), ('2023-08','2023-08-01','2023-09-01'),
    ('2023-09','2023-09-01','2023-10-01'), ('2023-10','2023-10-01','2023-11-01'),
    ('2023-11','2023-11-01','2023-12-01'), ('2023-12','2023-12-01','2024-01-01'),
    ('2024-01','2024-01-01','2024-02-01'), ('2024-02','2024-02-01','2024-03-01'),
    ('2024-03','2024-03-01','2024-04-01'), ('2024-04','2024-04-01','2024-05-01'),
    ('2024-05','2024-05-01','2024-06-01'), ('2024-06','2024-06-01','2024-07-01'),
]

print('Fetching monthly time series (18 months)...')
print('One GEE call per month — please wait ~2 minutes')
print()

ts_rows = []
for label, start, end in MONTHS:
    gc.collect()

    # FIRMS fire pixels
    firms_col = (
        ee.ImageCollection('FIRMS').filterDate(start, end)
        .filterBounds(ROI).select('T21')
    )
    n_imgs = firms_col.size().getInfo()
    fire_px = 0
    if n_imgs > 0:
        hot_px = (
            firms_col.max().gt(330)
            .reduceRegion(reducer=ee.Reducer.sum(),
                          geometry=ROI, scale=375, maxPixels=1e9, bestEffort=True)
            .getInfo()
        )
        fire_px = int(hot_px.get('T21', 0) or 0)

    # TROPOMI SO2 mean
    so2_col = (
        ee.ImageCollection('COPERNICUS/S5P/OFFL/L3_SO2')
        .filterDate(start, end).filterBounds(ROI)
        .select(['SO2_column_number_density', 'cloud_fraction'])
    )
    n_so2 = so2_col.size().getInfo()
    so2_mean = 0
    if n_so2 > 0:
        def mask_convert(img):
            cloud = img.select('cloud_fraction')
            so2   = img.select('SO2_column_number_density').multiply(2241.5).rename('SO2_DU')
            return so2.updateMask(cloud.lt(0.4))
        clean = so2_col.map(mask_convert)
        stats = clean.mean().reduceRegion(
            reducer=ee.Reducer.mean(), geometry=ROI,
            scale=5500, maxPixels=1e9, bestEffort=True).getInfo()
        so2_mean = stats.get('SO2_DU', 0) or 0

    row = {'month': label, 'fire_pixels': fire_px, 'so2_mean_du': round(so2_mean, 4)}
    ts_rows.append(row)
    print('  ' + label + ': fire_px=' + str(fire_px) + '  SO2=' + str(round(so2_mean,4)) + ' DU')

df_ts = pd.DataFrame(ts_rows)
df_ts['month'] = pd.to_datetime(df_ts['month'])
df_ts.to_csv(os.path.join(CACHE_DIR, 'm6_timeseries.csv'), index=False)
print()
print('Time series saved: data/cached/m6_timeseries.csv')


Fetching monthly time series (18 months)...
One GEE call per month — please wait ~2 minutes

  2023-01: fire_px=192  SO2=0.0667 DU
  2023-02: fire_px=989  SO2=-0.0445 DU
  2023-03: fire_px=957  SO2=-0.1225 DU
  2023-04: fire_px=333  SO2=-0.0587 DU
  2023-05: fire_px=244  SO2=-0.1578 DU
  2023-06: fire_px=0  SO2=-0.1227 DU
  2023-07: fire_px=56  SO2=-0.0777 DU
  2023-08: fire_px=0  SO2=0.0026 DU
  2023-09: fire_px=0  SO2=-0.0595 DU
  2023-10: fire_px=0  SO2=-0.0324 DU
  2023-11: fire_px=0  SO2=0.0015 DU
  2023-12: fire_px=0  SO2=0.0541 DU
  2024-01: fire_px=366  SO2=0.0327 DU
  2024-02: fire_px=1115  SO2=0.0092 DU
  2024-03: fire_px=1654  SO2=-0.2027 DU
  2024-04: fire_px=526  SO2=0.0817 DU
  2024-05: fire_px=65  SO2=-0.0261 DU
  2024-06: fire_px=0  SO2=0.0503 DU

Time series saved: data/cached/m6_timeseries.csv


## Cell 6 — Figure 3: Monthly time series chart (Plotly)

In [7]:
# Plotly only — no GEE calls
fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    subplot_titles=(
        'Monthly VIIRS fire pixel count (T21 > 330K)',
        'Monthly TROPOMI SO2 mean (Dobson Units) — OFFL product'
    ),
    vertical_spacing=0.1
)

# Split into baseline and recent for color coding
df_base = df_ts[df_ts['month'].dt.year == 2023]
df_rec  = df_ts[df_ts['month'].dt.year == 2024]

# Fire pixel time series
fig.add_trace(go.Scatter(
    x=df_base['month'], y=df_base['fire_pixels'],
    mode='lines+markers', name='2023 (baseline)',
    line=dict(color='#378ADD', width=2), marker=dict(size=7)
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=df_rec['month'], y=df_rec['fire_pixels'],
    mode='lines+markers', name='2024 (recent)',
    line=dict(color='#E24B4A', width=2), marker=dict(size=7)
), row=1, col=1)

# SO2 time series
fig.add_trace(go.Scatter(
    x=df_base['month'], y=df_base['so2_mean_du'],
    mode='lines+markers', name='2023 SO2',
    line=dict(color='#378ADD', width=2), marker=dict(size=7), showlegend=False
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=df_rec['month'], y=df_rec['so2_mean_du'],
    mode='lines+markers', name='2024 SO2',
    line=dict(color='#E24B4A', width=2), marker=dict(size=7), showlegend=False
), row=2, col=1)

# Dry season shading (Oct-Dec)
for yr in [2023, 2024]:
    fig.add_vrect(
        x0=str(yr)+'-10-01', x1=str(yr)+'-12-31',
        fillcolor='#FEF9E7', opacity=0.4, line_width=0,
        annotation_text='Dry season', annotation_position='top left',
        row=1, col=1
    )
    fig.add_vrect(
        x0=str(yr)+'-10-01', x1=str(yr)+'-12-31',
        fillcolor='#FEF9E7', opacity=0.4, line_width=0,
        row=2, col=1
    )

fig.add_hline(y=1.5, line_dash='dash', line_color='#854F0B',
              annotation_text='SO2 threshold (1.5 DU)', row=2, col=1)

fig.update_layout(
    title='PipelineWatch-NG — Monthly Sensor Time Series<br>'
          'Trans Niger Pipeline corridor | Jan 2023 – Jun 2024',
    height=600, plot_bgcolor='white', paper_bgcolor='white',
    legend=dict(x=0.01, y=0.99)
)
fig.update_yaxes(title_text='Hot pixels (T21 > 330K)', row=1, col=1, gridcolor='#f0f0f0')
fig.update_yaxes(title_text='SO2 mean (DU)',            row=2, col=1, gridcolor='#f0f0f0')
fig.update_xaxes(gridcolor='#f0f0f0')

out3 = os.path.join(OUTPUT_DIR, 'fig3_timeseries.html')
fig.write_html(out3)
print('Saved: ' + out3)
display(IFrame(src=out3, width='100%', height=630))


Saved: ../outputs\fig3_timeseries.html


## Cell 7 — Fetch cluster-level SO2 time series

GEE call only — samples SO2 at each confirmed cluster centroid monthly.
This shows whether the episodic SO2 signal is seasonal or year-round.

In [8]:
print('Fetching cluster-level SO2 time series...')
print('This takes ~3 minutes — one GEE call per cluster per month')
print()

cluster_ts = []
confirmed_clusters = df_clusters[df_clusters['SO2_confirmed'] == True]

for _, crow in confirmed_clusters.iterrows():
    cid = int(crow['cluster_id'])
    pt  = ee.Geometry.Point([crow['centroid_lon'], crow['centroid_lat']])
    buf = pt.buffer(10000)   # 10 km

    for label, start, end in MONTHS:
        gc.collect()
        so2_col = (
            ee.ImageCollection('COPERNICUS/S5P/OFFL/L3_SO2')
            .filterDate(start, end).filterBounds(buf)
            .select(['SO2_column_number_density', 'cloud_fraction'])
        )
        n = so2_col.size().getInfo()
        so2_mean = 0
        so2_max  = 0
        if n > 0:
            def mc(img):
                cloud = img.select('cloud_fraction')
                so2   = img.select('SO2_column_number_density').multiply(2241.5).rename('SO2_DU')
                return so2.updateMask(cloud.lt(0.4))
            clean = so2_col.map(mc)
            s = (
                clean.mean().addBands(clean.max().rename('SO2_max'))
                .reduceRegion(
                    reducer=ee.Reducer.mean().combine(ee.Reducer.max(), sharedInputs=False),
                    geometry=buf, scale=5500, maxPixels=1e6, bestEffort=True)
                .getInfo()
            )
            so2_mean = s.get('SO2_DU_mean', 0) or 0
            so2_max  = s.get('SO2_max_mean', 0) or 0
        cluster_ts.append({'cluster_id': cid, 'month': label,
                            'so2_mean': round(so2_mean,4), 'so2_max': round(so2_max,4)})

    print('  Cluster ' + str(cid) + ' done')

df_cluster_ts = pd.DataFrame(cluster_ts)
df_cluster_ts['month'] = pd.to_datetime(df_cluster_ts['month'])
df_cluster_ts.to_csv(os.path.join(CACHE_DIR, 'm6_cluster_timeseries.csv'), index=False)
print('Saved: data/cached/m6_cluster_timeseries.csv')


Fetching cluster-level SO2 time series...
This takes ~3 minutes — one GEE call per cluster per month

  Cluster 0 done
  Cluster 1 done
  Cluster 2 done
  Cluster 4 done
  Cluster 5 done
  Cluster 6 done
Saved: data/cached/m6_cluster_timeseries.csv


## Cell 8 — Figure 4: Per-cluster SO2 time series (Plotly)

In [9]:
# Plotly only — no GEE calls
confirmed_ids = df_clusters[df_clusters['SO2_confirmed'] == True]['cluster_id'].tolist()
n_conf = len(confirmed_ids)

ncols  = 2
nrows  = (n_conf + 1) // ncols
fig    = make_subplots(
    rows=nrows, cols=ncols,
    subplot_titles=['Cluster ' + str(int(cid)) for cid in confirmed_ids],
    shared_yaxes=True, vertical_spacing=0.12
)

colors = ['#E24B4A','#378ADD','#1D9E75','#EF9F27','#7F77DD','#D85A30']

for i, cid in enumerate(confirmed_ids):
    row_n = i // ncols + 1
    col_n = i % ncols + 1
    g     = df_cluster_ts[df_cluster_ts['cluster_id'] == cid]
    color = colors[i % len(colors)]

    # SO2 max as shaded area
    fig.add_trace(go.Scatter(
        x=pd.concat([g['month'], g['month'].iloc[::-1]]),
        y=pd.concat([g['so2_max'], g['so2_mean'].iloc[::-1]]),
        fill='toself', fillcolor=color, opacity=0.15,
        line=dict(width=0), showlegend=False, name='SO2 range'
    ), row=row_n, col=col_n)

    # SO2 mean line
    fig.add_trace(go.Scatter(
        x=g['month'], y=g['so2_mean'],
        mode='lines+markers', name='SO2 mean C'+str(int(cid)),
        line=dict(color=color, width=2), marker=dict(size=5)
    ), row=row_n, col=col_n)

    # Threshold line
    fig.add_hline(y=1.5, line_dash='dash', line_color='#854F0B',
                  line_width=1, row=row_n, col=col_n)

    # Dry season shading
    for yr in [2023, 2024]:
        fig.add_vrect(
            x0=str(yr)+'-10-01', x1=str(yr)+'-12-31',
            fillcolor='#FEF9E7', opacity=0.4, line_width=0,
            row=row_n, col=col_n
        )

fig.update_layout(
    title='PipelineWatch-NG — Per-Cluster SO2 Time Series (TROPOMI OFFL)<br>'
          'Shaded band = mean to max  |  Yellow = dry season  |  Dashed = 1.5 DU threshold',
    height=200 * nrows + 100,
    plot_bgcolor='white', paper_bgcolor='white',
    showlegend=False
)
fig.update_yaxes(title_text='SO2 (DU)', gridcolor='#f0f0f0')
fig.update_xaxes(gridcolor='#f0f0f0')

out4 = os.path.join(OUTPUT_DIR, 'fig4_cluster_timeseries.html')
fig.write_html(out4)
print('Saved: ' + out4)
display(IFrame(src=out4, width='100%', height=min(900, 200*nrows+150)))


Saved: ../outputs\fig4_cluster_timeseries.html


## Cell 9 — Export figures for paper submission

Saves all figures at 300 DPI (journal standard) and 600 DPI (print quality).

In [11]:
# The static maps (fig1, fig2) are already saved as PNG.
# This cell saves the Plotly charts as static PNG using kaleido.
# Install: pip install kaleido
!pip install kaleido
try:
    import kaleido
    fig3_static = os.path.join(OUTPUT_DIR, 'fig3_timeseries_300dpi.png')
    fig.write_image(fig3_static, width=1800, height=900, scale=2)
    print('Saved static PNG: ' + fig3_static)
except ImportError:
    print('kaleido not installed — Plotly charts saved as HTML only')
    print('To export static PNG: pip install kaleido')
    print('Then re-run this cell.')

print()

print('=== Module 6 complete ===')
print('Fig 1 (study area)     : outputs/fig1_study_area.html')
print('Fig 2 (cluster detail) : outputs/fig2_cluster_detail.html')
print('Fig 3 (time series)    : outputs/fig3_timeseries.html')
print('Fig 4 (cluster SO2)    : outputs/fig4_cluster_timeseries.html')
print()
print('To export static PNG for paper submission:')
print('  pip install kaleido')
print('  Then re-run this cell')


  Using cached kaleido-1.2.0-py3-none-any.whl.metadata (5.6 kB)
  Using cached choreographer-1.2.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached logistro-2.0.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached pytest_timeout-2.4.0-py3-none-any.whl.metadata (20 kB)
Using cached kaleido-1.2.0-py3-none-any.whl (68 kB)
Using cached choreographer-1.2.1-py3-none-any.whl (49 kB)
Using cached logistro-2.0.1-py3-none-any.whl (8.6 kB)
Using cached pytest_timeout-2.4.0-py3-none-any.whl (14 kB)

   ---------------------------------------- 0/6 [simplejson]
   ---------------------------------------- 0/6 [simplejson]
   -------------------------- ------------- 4/6 [choreographer]
   --------------------------------- ------ 5/6 [kaleido]
   ---------------------------------------- 6/6 [kaleido]

Saved static PNG: ../outputs\fig3_timeseries_300dpi.png

=== Module 6 complete ===
Fig 1 (study area)     : outputs/fig1_study_area.html
Fig 2 (cluster detail) : outputs/fig2_cluster_detail.html
Fig 3 (ti